# Quality-safe latency-aware LLM routing on Colab

## What this notebook teaches

This is both an executable experiment and a guided walkthrough. By the end,
you should be able to explain:

1. how warm latency and deterministic quality are measured fairly;
2. how observed candidate outcomes become per-model **quality-safety** labels;
3. why LoRA is used to adapt ModernBERT with little trainable state;
4. how the router predicts safety, latency, and output length;
5. why calibration and checkpoint selection use validation data only; and
6. how the sealed test result is compared with strongest, fastest, and oracle
   baselines.

**Research question.** Can a prompt-only router select the lowest-latency
candidate predicted to be quality-safe, retain at least 98% of the strongest
model's mean accuracy, and produce a statistically credible latency reduction?

```mermaid
flowchart LR
    A["Reference-scored prompts"] --> B["Run every candidate"]
    B --> C["Quality + latency + token labels"]
    C --> D["ModernBERT + LoRA"]
    D --> E["Safety / latency / token heads"]
    E --> F["Validation calibration"]
    F --> G["Fastest predicted-safe model"]
    G --> H["Sealed test evaluation"]
```

Candidate pool:

- `Qwen/Qwen2.5-7B-Instruct` in 4-bit: strong fallback;
- `Qwen/Qwen2.5-1.5B-Instruct`: small autoregressive control;
- `Efficient-Large-Model/Fast_dLLM_v2_1.5B`: block-diffusion candidate.

The synchronized default uses 300 prompts from each of GSM8K, MMLU, and
ARC-Challenge: 900 prompts and 2,700 candidate generations.

### How to run it

1. Select a GPU Colab runtime.
2. Run the install cell once.
3. Restart the runtime so the pinned packages are active.
4. Run the notebook from top to bottom.

Long generation stages are resumable through fingerprinted Google Drive
caches. The scope document is `llm_router_project_scope_3.md`.

In [13]:
# Gradio is unused and its preinstalled dependency set conflicts with the
# Fast-dLLM-compatible Hugging Face Hub client.
%pip uninstall -q -y gradio gradio-client
%pip install -q -U "transformers==4.53.1" \
    "huggingface-hub==0.36.2" accelerate "bitsandbytes>=0.46" \
    "datasets>=3.0" pyarrow scikit-learn einops "peft==0.17.1" "ipywidgets>=8"

## 1. Synchronized experiment contract

### Why start with a contract?

Latency experiments become unreliable when a resumed session silently changes
a model revision, prompt template, output cap, or GPU. The configuration cell
therefore defines the experiment before downloading data or models.

It separates two identities:

- the **generation fingerprint**, which determines whether expensive candidate
  responses can be reused; and
- the **router fingerprint**, which identifies the LoRA and training setup.

V2 generations cannot enter this experiment because v3 uses a different Drive
root and JSON-only prompt contract. The cell also verifies that each Fast-dLLM
output cap is an exact multiple of its 32-token block size.

**Expected output:** GPU and package information, resolved immutable model
revisions, and short generation/router fingerprint strings.

In [21]:
from __future__ import annotations

import gc, hashlib, html as html_lib, json, math, os, random, re, time, warnings
from dataclasses import asdict, dataclass
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from IPython.display import HTML, clear_output, display
from sklearn.metrics import brier_score_loss, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, WeightedRandomSampler

# --- Experiment identity and dataset size (must match the v3 scope) ---
SCOPE_SCHEMA_VERSION = 3
SCOPE_FILE_NAME = "llm_router_project_scope_3.md"
SEED = 42
N_PER_TASK = 300
TASKS = ("gsm8k", "mmlu", "arc_challenge")
TASK_MAX_NEW_TOKENS = {"gsm8k": 192, "mmlu": 64, "arc_challenge": 64}
MAX_INPUT_TOKENS = 1024
WARMUP_PROMPTS = 2
FLUSH_EVERY = 5
MIN_QUALITY_RETENTION = 0.98
QUALITY_SAFETY_EPSILON = 0.0
USE_GOOGLE_DRIVE = True
RUN_FAST_DLLM_REPAIR_PILOT = True
FAST_REPAIR_PROMPTS_PER_TASK = 5
FAST_AUDIT_SAMPLE_PER_TASK = 5
RUN_EXPANDED_BENCHMARK = True

# --- Parameter-efficient router adaptation ---
# ModernBERT stays frozen; LoRA adds small trainable low-rank matrices to
# every linear layer. The three prediction heads are also trainable.
ROUTER_ENCODER_REPO = "nomic-ai/modernbert-embed-base"
ROUTER_ENCODER_REVISION = "d556a88e332558790b210f7bdbe87da2fa94a8d8"
ROUTER_USE_LORA = True
ROUTER_LORA_R = 8
ROUTER_LORA_ALPHA = 16
ROUTER_LORA_DROPOUT = 0.05
ROUTER_LORA_TARGET_MODULES = "all-linear"
ROUTER_BATCH_SIZE = 4
ROUTER_GRADIENT_ACCUMULATION = 4
ROUTER_MAX_EPOCHS = 15
ROUTER_MIN_EPOCHS = 8
ROUTER_EARLY_STOPPING_PATIENCE = 6
ROUTER_LORA_LR = 1e-4
ROUTER_HEAD_LR = 2e-4
ROUTER_WEIGHT_DECAY = 0.01
ROUTER_WARMUP_RATIO = 0.05
ROUTER_MAX_GRAD_NORM = 1.0
ROUTER_SAFETY_LOSS_WEIGHT = 1.0
ROUTER_LATENCY_LOSS_WEIGHT = 0.75
ROUTER_TOKEN_LOSS_WEIGHT = 0.10
ROUTER_WARMUP_PROMPTS = 2
SAFETY_THRESHOLD_GRID = (0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.975)

SYSTEM_PROMPT = (
    "You are a careful benchmark assistant. Think through the problem "
    "internally and return only the requested JSON object, with no prose."
)
PROMPT_TEMPLATE_VERSION = "v3-json-only-2026-08-07"
FAST_DLLM_REVISION = "25093b6f63300adfd57f72145083c8a528fe4f16"
FAST_DLLM_PARAMETERS = {"block_size": 32, "small_block_size": 8, "threshold": 0.9}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert SCOPE_SCHEMA_VERSION == 3
assert N_PER_TASK == 300 and len(TASKS) == 3
assert TASK_MAX_NEW_TOKENS == {"gsm8k": 192, "mmlu": 64, "arc_challenge": 64}
assert all(
    limit % FAST_DLLM_PARAMETERS["block_size"] == 0
    for limit in TASK_MAX_NEW_TOKENS.values()
)
assert ROUTER_USE_LORA
assert ROUTER_LORA_R == 8 and ROUTER_LORA_ALPHA == 16
assert ROUTER_LORA_TARGET_MODULES == "all-linear"
assert torch.cuda.is_available(), "Choose Runtime > Change runtime type > GPU in Colab."
assert transformers.__version__ == "4.53.1", (
    f"Expected transformers 4.53.1, found {transformers.__version__}. "
    "Restart the Colab runtime after the install cell."
)

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/llm_router_v3")
else:
    ROOT = Path("/content/llm_router_v3")

for folder in [ROOT / "data", ROOT / "cache", ROOT / "reports", ROOT / "artifacts"]:
    folder.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = "/content/hf_cache"
GPU_NAME = torch.cuda.get_device_name(0)
COMPUTE_DTYPE = (
    torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
)
print({"gpu": GPU_NAME, "dtype": str(COMPUTE_DTYPE), "root": str(ROOT)})
print({"transformers": transformers.__version__, "torch": torch.__version__})
print({
    "lora_rank": ROUTER_LORA_R, "lora_alpha": ROUTER_LORA_ALPHA,
    "lora_dropout": ROUTER_LORA_DROPOUT,
    "lora_targets": ROUTER_LORA_TARGET_MODULES,
})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'gpu': 'Tesla T4', 'dtype': 'torch.float16', 'root': '/content/drive/MyDrive/llm_router_v3'}
{'transformers': '4.53.1', 'torch': '2.11.0+cu128'}
{'lora_rank': 8, 'lora_alpha': 16, 'lora_dropout': 0.05, 'lora_targets': 'all-linear'}


In [15]:
from huggingface_hub import HfApi

@dataclass(frozen=True)
class ModelSpec:
    repo: str
    kind: str
    four_bit: bool = False
    pinned_revision: str | None = None

MODELS = {
    "qwen2.5-1.5b-ar": ModelSpec("Qwen/Qwen2.5-1.5B-Instruct", "causal"),
    "fast-dllm-v2-1.5b": ModelSpec(
        "Efficient-Large-Model/Fast_dLLM_v2_1.5B", "fast_dllm",
        pinned_revision=FAST_DLLM_REVISION,
    ),
    "qwen2.5-7b-4bit": ModelSpec(
        "Qwen/Qwen2.5-7B-Instruct", "causal", four_bit=True
    ),
}
MODELS_TO_RUN = list(MODELS)  # Set to one key to split generation across sessions.

# Resolve floating Hub branches once, then record the exact commit SHA.
api = HfApi()
MODEL_REVISIONS = {
    name: spec.pinned_revision or api.model_info(spec.repo).sha
    for name, spec in MODELS.items()
}

CACHE_CONTRACT = {
    "schema_version": SCOPE_SCHEMA_VERSION,
    "seed": SEED,
    "tasks": TASKS,
    "task_max_new_tokens": TASK_MAX_NEW_TOKENS,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "prompt_template_version": PROMPT_TEMPLATE_VERSION,
    "system_prompt": SYSTEM_PROMPT,
    "models": {name: asdict(spec) for name, spec in MODELS.items()},
    "model_revisions": MODEL_REVISIONS,
    "fast_dllm_parameters": FAST_DLLM_PARAMETERS,
    "transformers": transformers.__version__,
    "torch": torch.__version__,
    "gpu": GPU_NAME,
    "dtype": str(COMPUTE_DTYPE),
}
# Candidate generations and router training have separate fingerprints.
# This lets us reuse expensive generations while comparing router variants.
RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(CACHE_CONTRACT, sort_keys=True).encode()
).hexdigest()
ROUTER_CONTRACT = {
    "encoder_repo": ROUTER_ENCODER_REPO,
    "encoder_revision": ROUTER_ENCODER_REVISION,
    "adaptation": "lora",
    "lora_r": ROUTER_LORA_R,
    "lora_alpha": ROUTER_LORA_ALPHA,
    "lora_dropout": ROUTER_LORA_DROPOUT,
    "lora_target_modules": ROUTER_LORA_TARGET_MODULES,
    "lora_lr": ROUTER_LORA_LR,
    "head_lr": ROUTER_HEAD_LR,
    "loss_weights": {
        "safety": ROUTER_SAFETY_LOSS_WEIGHT,
        "latency": ROUTER_LATENCY_LOSS_WEIGHT,
        "tokens": ROUTER_TOKEN_LOSS_WEIGHT,
    },
}
ROUTER_FINGERPRINT = hashlib.sha256(
    json.dumps(ROUTER_CONTRACT, sort_keys=True).encode()
).hexdigest()
RUN_MANIFEST = {
    **CACHE_CONTRACT,
    "scope_file": SCOPE_FILE_NAME,
    "n_per_task": N_PER_TASK,
    "run_fingerprint": RUN_FINGERPRINT,
    "router_contract": ROUTER_CONTRACT,
    "router_fingerprint": ROUTER_FINGERPRINT,
    "minimum_quality_retention": MIN_QUALITY_RETENTION,
    "quality_safety_epsilon": QUALITY_SAFETY_EPSILON,
    "selection_rule": "lowest predicted latency among calibrated quality-safe candidates",
}

manifest_path = ROOT / "run_manifest_v3.json"
if manifest_path.exists():
    previous = json.loads(manifest_path.read_text())
    assert previous["schema_version"] == SCOPE_SCHEMA_VERSION
    assert previous["run_fingerprint"] == RUN_FINGERPRINT, (
        "The v3 cache contract changed. Use a new ROOT rather than mixing measurements."
    )
    assert int(previous["n_per_task"]) <= N_PER_TASK, (
        "Reducing N_PER_TASK in an existing v3 ROOT is not supported."
    )
manifest_path.write_text(json.dumps(RUN_MANIFEST, indent=2), encoding="utf-8")
(ROOT / "reports" / "synchronized_contract.json").write_text(
    json.dumps(RUN_MANIFEST, indent=2), encoding="utf-8"
)

display(pd.DataFrame({
    name: {**asdict(spec), "resolved_revision": MODEL_REVISIONS[name]}
    for name, spec in MODELS.items()
}).T)
print("Generation fingerprint:", RUN_FINGERPRINT[:16])
print("Router fingerprint:", ROUTER_FINGERPRINT[:16])

,repo,kind,four_bit,pinned_revision,resolved_revision
qwen2.5-1.5b-ar,Qwen/Qwen2.5-1.5B-Instruct,causal,False,None,989aa7980e4cf806f80c7fef2b1adb7bc71aa306
fast-dllm-v2-1.5b,Efficient-Large-Model/Fast_dLLM_v2_1.5B,fast_dllm,False,25093b6f63300adfd57f72145083c8a528fe4f16,25093b6f63300adfd57f72145083c8a528fe4f16
qwen2.5-7b-4bit,Qwen/Qwen2.5-7B-Instruct,causal,True,None,a09a35458c702b33eeacc393d103063234e8bc28


Generation fingerprint: d2044a470f45f714
Router fingerprint: 1e74d980bca9ca0a


## 2. Reference-scored data and routing metadata

### Why these datasets?

GSM8K, MMLU, and ARC-Challenge have public references that can score newly
generated answers deterministically. RouterBench contains outcomes for older
model responses, but not the references needed for this new candidate pool.

The notebook converts each example into a common record:

| Field | Purpose |
|---|---|
| `prompt_id` | Stable hash used by resumable caches |
| `task` / `subject` | Helps the prompt-only router learn specialization |
| `num_choices` | Separates multiple-choice structure from free-form math |
| `prompt_words` / `length_bin` | Provides a coarse complexity and latency clue |
| `reference` | Deterministic quality target |

Sampling is deterministic. Increasing `N_PER_TASK` retains existing prompt IDs
and appends prompts in a reproducible order.

**Expected output:** exactly 300 rows per task and a preview of normalized
prompts and references.

In [16]:
from datasets import load_dataset

LABELS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def stable_id(task: str, prompt: str) -> str:
    return hashlib.sha1(f"{task}\0{prompt}".encode()).hexdigest()[:16]

def add_prompt_metadata(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["prompt_words"] = result.prompt.str.split().str.len().astype(int)
    bins = [0, 64, 128, 256, 512, np.inf]
    labels = ["xs", "s", "m", "l", "xl"]
    result["length_bin"] = pd.cut(
        result.prompt_words, bins=bins, labels=labels, include_lowest=True
    ).astype(str)
    return result

def build_prompt_pool() -> pd.DataFrame:
    # Build the full labeled source pool first; deterministic sampling happens later.
    rows = []

    gsm = load_dataset("openai/gsm8k", "main", split="test").to_pandas()
    for row in gsm.itertuples():
        prompt = (
            'Solve the problem carefully. Think through the calculation internally. '
            'Return only JSON {"answer": "number"}.\n\n'
            f"{row.question}"
        )
        numbers = re.findall(r"-?\d[\d,]*(?:\.\d+)?", str(row.answer))
        reference = canonical_number(numbers[-1])
        rows.append({
            "task": "gsm8k", "subject": "grade_school_math", "num_choices": 0,
            "prompt": prompt, "reference": reference,
        })

    mmlu = load_dataset("cais/mmlu", "all", split="test").to_pandas()
    for row in mmlu.itertuples():
        choices = [str(value) for value in row.choices]
        rendered = "\n".join(f"{LABELS[i]}. {text}" for i, text in enumerate(choices))
        prompt = (
            'Choose the best answer. Think through it internally. '
            'Return only JSON {"answer": "LETTER"}.\n\n'
            f"Question: {row.question}\n{rendered}"
        )
        rows.append({
            "task": "mmlu", "subject": str(getattr(row, "subject", "unknown")),
            "num_choices": len(choices), "prompt": prompt,
            "reference": LABELS[int(row.answer)],
        })

    arc = load_dataset(
        "allenai/ai2_arc", "ARC-Challenge", split="test"
    ).to_pandas()
    for row in arc.itertuples():
        choice_labels = [str(value) for value in row.choices["label"]]
        choice_text = [str(value) for value in row.choices["text"]]
        answer_key = str(row.answerKey)
        if answer_key not in choice_labels:
            raise ValueError(
                f"ARC answer key {answer_key!r} is absent from {choice_labels!r}."
            )
        answer_index = choice_labels.index(answer_key)
        rendered = "\n".join(
            f"{LABELS[i]}. {text}" for i, text in enumerate(choice_text)
        )
        prompt = (
            'Choose the best answer. Think through it internally. '
            'Return only JSON {"answer": "LETTER"}.\n\n'
            f"Question: {row.question}\n{rendered}"
        )
        rows.append({
            "task": "arc_challenge", "subject": "science",
            "num_choices": len(choice_text), "prompt": prompt,
            "reference": LABELS[answer_index],
        })

    result = pd.DataFrame(rows)
    result.insert(0, "prompt_id", [
        stable_id(task, prompt) for task, prompt in zip(result.task, result.prompt)
    ])
    conflicting = result.groupby("prompt_id").reference.nunique().gt(1)
    assert not conflicting.any()
    result = result.drop_duplicates("prompt_id", keep="first").reset_index(drop=True)
    return add_prompt_metadata(result)

def extend_prompts(existing: pd.DataFrame, n_per_task: int) -> pd.DataFrame:
    # Existing IDs are immutable so a disconnected Colab session can resume safely.
    required = {
        "prompt_id", "task", "subject", "num_choices", "prompt_words",
        "length_bin", "prompt", "reference",
    }
    if not existing.empty:
        missing = required - set(existing.columns)
        if missing:
            raise ValueError(f"Stored v3 prompts are missing {sorted(missing)}")
        if existing.groupby("task").size().gt(n_per_task).any():
            raise ValueError("Stored prompts exceed N_PER_TASK; use a new ROOT.")

    pool = build_prompt_pool()
    if not existing.empty:
        known_ids = set(pool.prompt_id)
        stale = set(existing.prompt_id) - known_ids
        if stale:
            raise ValueError("Stored prompt IDs no longer match the source datasets.")

    parts = []
    task_seeds = {"gsm8k": SEED, "mmlu": SEED + 1, "arc_challenge": SEED + 2}
    for task, task_seed in task_seeds.items():
        kept = existing.loc[existing.task.eq(task)].copy()
        needed = n_per_task - len(kept)
        candidates = pool.loc[
            pool.task.eq(task) & ~pool.prompt_id.isin(set(existing.prompt_id))
        ].copy()
        candidates["_priority"] = [
            hashlib.sha1(f"{task_seed}\0{prompt_id}".encode()).hexdigest()
            for prompt_id in candidates.prompt_id
        ]
        additions = candidates.sort_values("_priority").head(needed).drop(columns="_priority")
        if len(additions) != needed:
            raise ValueError(f"Not enough {task} prompts for N_PER_TASK={n_per_task}.")
        parts.extend([kept, additions])

    result = pd.concat(parts, ignore_index=True)
    assert len(result) == len(TASKS) * n_per_task and result.prompt_id.is_unique
    return result

def canonical_number(value: str | None) -> str | None:
    if value is None:
        return None
    try:
        number = Decimal(str(value).replace(",", "").strip())
    except InvalidOperation:
        return None
    if not number.is_finite():
        return None
    if number == 0:
        return "0"
    return format(number.normalize(), "f")

prompts_path = ROOT / "data" / "prompts_v3.parquet"
stored_prompts = (
    pd.read_parquet(prompts_path) if prompts_path.exists()
    else pd.DataFrame(columns=[
        "prompt_id", "task", "subject", "num_choices", "prompt_words",
        "length_bin", "prompt", "reference",
    ])
)
counts = stored_prompts.groupby("task").size() if len(stored_prompts) else pd.Series(dtype=int)
if any(int(counts.get(task, 0)) != N_PER_TASK for task in TASKS):
    prompts = extend_prompts(stored_prompts, N_PER_TASK)
    prompts.to_parquet(prompts_path, index=False)
else:
    prompts = stored_prompts

display(prompts.groupby("task").agg(
    prompts=("prompt_id", "size"), mean_words=("prompt_words", "mean")
).round(1))
display(prompts.head(2))

,prompts,mean_words
task,,
arc_challenge,300,62.5
gsm8k,300,59.7
mmlu,300,100.3


,prompt_id,task,subject,num_choices,prompt_words,length_bin,prompt,reference
0,072cdff511c82369,gsm8k,grade_school_math,0,44,xs,Solve the problem carefully. Think through the...,75
1,1bd27ad845bdd077,gsm8k,grade_school_math,0,101,s,Solve the problem carefully. Think through the...,20


## 3. Revision-pinned model adapters

A model adapter gives all candidates the same interface while preserving the
decoding algorithm appropriate to each architecture.

### Fair timing boundary

```text
prompt text --(encode_s, reported separately)--> token IDs
token IDs   --(generation_s, synchronized)-----> generated token IDs
generated IDs --(outside timing)---------------> decoded response text
```

`generation_s` includes prefill and token generation but excludes model loading,
prompt tokenization, and text decoding. `torch.cuda.synchronize()` prevents the
asynchronous GPU queue from making measurements appear artificially short.

Qwen uses deterministic greedy generation. Fast-dLLM uses its official custom
`generate()` with explicit 32-token blocks, 8-token sub-blocks, and threshold
0.9. All candidates share the same task-specific maximum output length.

In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

class LocalAdapter:
    def __init__(self, name: str, spec: ModelSpec):
        self.name, self.spec = name, spec
        self.revision = MODEL_REVISIONS[name]
        # Quantization is used only for the 7B fallback so it fits on a T4.
        quant = None
        if spec.four_bit:
            quant = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=COMPUTE_DTYPE,
                bnb_4bit_use_double_quant=True,
            )

        started = time.perf_counter()
        is_fast = spec.kind == "fast_dllm"
        self.codec = AutoTokenizer.from_pretrained(
            spec.repo, revision=self.revision, trust_remote_code=is_fast
        )
        load_kwargs = {
            "device_map": "auto", "torch_dtype": COMPUTE_DTYPE,
            "trust_remote_code": is_fast, "revision": self.revision,
            "low_cpu_mem_usage": True,
        }
        if quant is not None:
            load_kwargs["quantization_config"] = quant
        self.model = AutoModelForCausalLM.from_pretrained(spec.repo, **load_kwargs)
        self.model.eval()
        self.load_time_s = time.perf_counter() - started

    def encode(self, prompt: str):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
        text = self.codec.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        encoded = self.codec(
            text, return_tensors="pt", truncation=True,
            max_length=MAX_INPUT_TOKENS,
        )
        return {key: value.to(self.model.device) for key, value in encoded.items()}

    @torch.inference_mode()
    def generate(self, prompt: str, task: str) -> dict:
        encode_started = time.perf_counter()
        inputs = self.encode(prompt)
        encode_s = time.perf_counter() - encode_started
        input_tokens = int(inputs["input_ids"].shape[1])
        max_new_tokens = TASK_MAX_NEW_TOKENS[task]

        # Synchronize immediately before and after generation: CUDA kernels are
        # asynchronous, so wall time is otherwise understated.
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        started = time.perf_counter()
        if self.spec.kind == "fast_dllm":
            output = self.model.generate(
                inputs["input_ids"], tokenizer=self.codec,
                max_new_tokens=max_new_tokens, **FAST_DLLM_PARAMETERS,
            )
        else:
            output = self.model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=getattr(self.codec, "pad_token_id", None),
            )
        torch.cuda.synchronize()
        generation_s = time.perf_counter() - started

        new_ids = output[0, input_tokens:]
        response = self.codec.decode(new_ids, skip_special_tokens=True).strip()
        output_tokens = int(new_ids.numel())
        return {
            "response": response, "encode_s": encode_s,
            "generation_s": generation_s, "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "tokens_per_second": output_tokens / max(generation_s, 1e-9),
            "peak_vram_gb": torch.cuda.max_memory_allocated() / 2**30,
        }

def unload(adapter):
    if hasattr(adapter, "model"):
        del adapter.model
    if hasattr(adapter, "codec"):
        del adapter.codec
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

## 4. Fingerprinted, resumable warm benchmarking

Model generation is the expensive stage, so every successful prompt-model row
is durable. The cache filename contains the generation fingerprint, and each
row repeats it for defense in depth.

For each model the lifecycle is:

1. load the tokenizer and model once;
2. run one smoke test;
3. run two unmeasured warm-up prompts;
4. generate and synchronize one prompt at a time;
5. atomically save every five new rows; and
6. unload the model and clear CUDA memory.

Fast-dLLM first runs a five-prompt-per-task repair pilot. Expansion stops if
parser coverage, degeneracy, or token-cap checks fail. This prevents spending
hours collecting measurements from a broken custom runtime.

In [18]:
CACHE_TAG = RUN_FINGERPRINT[:16]

def cache_path(model_name: str) -> Path:
    return ROOT / "cache" / f"{model_name}__{CACHE_TAG}.parquet"

# Strict formatting and answer correctness are related but distinct signals.
def strict_json_answer(response: str) -> str | None:
    try:
        payload = json.loads(str(response).strip())
    except (json.JSONDecodeError, TypeError):
        return None
    if not isinstance(payload, dict) or set(payload) != {"answer"}:
        return None
    return str(payload["answer"]).strip()

def extract_answer(response: str, task: str) -> str | None:
    text = str(response)
    strict = strict_json_answer(text)
    json_answers = re.findall(
        r'["\']answer["\']\s*:\s*["\']?([^"\'\n}\s]+)', text, flags=re.I
    )
    if task == "gsm8k":
        candidates = ([strict] if strict is not None else json_answers)
        if not candidates:
            candidates = re.findall(
                r"[-+]?\d[\d,]*(?:\.\d+)?(?:[eE][-+]?\d+)?", text
            )
        return canonical_number(candidates[-1]) if candidates else None

    candidate = strict or (json_answers[-1] if json_answers else None)
    if candidate is not None:
        match = re.search(r"[A-Z]", candidate.upper())
        if match:
            return match.group(0)
    choices = re.findall(r"(?:^|\W)([A-F])(?:\W|$)", text.upper())
    return choices[-1] if choices else None

def is_degenerate_response(response: str) -> bool:
    words = re.findall(r"\w+", str(response).lower())
    lines = [line.strip().lower() for line in str(response).splitlines() if line.strip()]
    low_diversity = len(words) >= 24 and len(set(words)) / len(words) < 0.20
    repeated_lines = len(lines) >= 8 and len(set(lines)) / len(lines) < 0.40
    return low_diversity or repeated_lines

def save_cache(frame: pd.DataFrame, path: Path) -> None:
    clean = frame.drop_duplicates(["prompt_id", "model"], keep="last")
    temporary = path.with_suffix(".tmp.parquet")
    clean.to_parquet(temporary, index=False)
    temporary.replace(path)

def audit_fast_dllm_cache(sample_per_task: int = FAST_AUDIT_SAMPLE_PER_TASK):
    path = cache_path("fast-dllm-v2-1.5b")
    if not path.exists():
        print("No fingerprint-matching Fast-dLLM cache exists.")
        return pd.DataFrame(), False

    cached = pd.read_parquet(path).drop_duplicates(["prompt_id", "model"], keep="last")
    required = {"prompt_id", "task", "model", "status", "response", "run_fingerprint"}
    missing = required - set(cached.columns)
    if missing:
        raise ValueError(f"Fast-dLLM cache is missing {sorted(missing)}")
    cached = cached.loc[
        cached.status.eq("ok") & cached.run_fingerprint.eq(RUN_FINGERPRINT)
    ].copy()
    audit = cached.merge(
        prompts[["prompt_id", "task", "prompt", "reference"]],
        on=["prompt_id", "task"], how="inner", validate="one_to_one",
    )
    if audit.empty:
        return audit, False

    audit["prediction"] = [
        extract_answer(response, task)
        for response, task in zip(audit.response, audit.task)
    ]
    audit["parsed"] = audit.prediction.notna()
    audit["strict_format"] = audit.response.map(strict_json_answer).notna()
    audit["quality"] = audit.prediction.eq(audit.reference)
    audit["degenerate_response"] = audit.response.map(is_degenerate_response)
    audit["at_token_cap"] = [
        tokens >= TASK_MAX_NEW_TOKENS[task]
        for tokens, task in zip(audit.output_tokens, audit.task)
    ]
    summary = audit.groupby("task").agg(
        prompts=("prompt_id", "size"), parse_rate=("parsed", "mean"),
        strict_format_rate=("strict_format", "mean"),
        exact_match=("quality", "mean"),
        degenerate_rate=("degenerate_response", "mean"),
        token_cap_rate=("at_token_cap", "mean"),
        mean_latency_s=("generation_s", "mean"),
    )
    display(summary.round(3))

    checks = pd.Series({
        "all_tasks_present": set(audit.task) == set(TASKS),
        "parse_rate_at_least_90pct": audit.parsed.mean() >= 0.90,
        "at_least_one_exact_match": bool(audit.quality.any()),
        "no_degenerate_responses": not bool(audit.degenerate_response.any()),
        "token_cap_rate_at_most_25pct": audit.at_token_cap.mean() <= 0.25,
    }, name="passed")
    display(checks.rename_axis("audit_check").to_frame())
    passed = bool(checks.all())

    sample_count = min(sample_per_task, int(audit.groupby("task").size().min()))
    sampled = audit.groupby("task", group_keys=False).sample(
        n=sample_count, random_state=SEED
    )
    with pd.option_context("display.max_colwidth", 1000, "display.max_rows", None):
        display(sampled[[
            "task", "prompt_id", "reference", "prediction", "parsed",
            "strict_format", "quality", "output_tokens", "generation_s", "response",
        ]].sort_values(["task", "prompt_id"]))
    audit.to_parquet(ROOT / "reports" / "fast_dllm_audit_v3.parquet", index=False)
    print("Audit recommendation:", "PASS" if passed else "HOLD")
    return audit, passed

In [19]:
def benchmark_model(
    model_name: str, prompt_frame: pd.DataFrame | None = None
) -> pd.DataFrame:
    target_prompts = prompts if prompt_frame is None else prompt_frame
    path = cache_path(model_name)
    cached = pd.read_parquet(path) if path.exists() else pd.DataFrame()
    if cached.empty:
        completed = set()
    else:
        required = {"prompt_id", "model", "status", "run_fingerprint"}
        missing = required - set(cached.columns)
        if missing:
            raise ValueError(f"Cache {path} is missing {sorted(missing)}")
        successful = cached.status.eq("ok") & cached.run_fingerprint.eq(RUN_FINGERPRINT)
        completed = set(cached.loc[successful, "prompt_id"])

    remaining = target_prompts.loc[~target_prompts.prompt_id.isin(completed)]
    print(f"{model_name}: {len(completed)} cached, {len(remaining)} remaining")
    if remaining.empty:
        return cached

    # The adapter is deliberately created outside the prompt loop: loading a
    # model per request would turn this into a cold-start benchmark.
    adapter = None
    try:
        adapter = LocalAdapter(model_name, MODELS[model_name])
        smoke_row = remaining.iloc[0]
        smoke = adapter.generate(smoke_row.prompt, smoke_row.task)
        assert smoke["response"] and smoke["output_tokens"] > 0
        if model_name == "fast-dllm-v2-1.5b" and is_degenerate_response(smoke["response"]):
            raise RuntimeError("Fast-dLLM smoke test produced a degenerate response.")
        print("smoke:", smoke["response"][:240].replace("\n", " "))

        for row in target_prompts.head(WARMUP_PROMPTS).itertuples():
            adapter.generate(row.prompt, row.task)

        new_rows = []
        for run_index, row in enumerate(remaining.itertuples(), 1):
            base = {
                "prompt_id": row.prompt_id, "task": row.task,
                "model": model_name, "model_repo": MODELS[model_name].repo,
                "model_revision": MODEL_REVISIONS[model_name], "gpu": GPU_NAME,
                "load_time_s": adapter.load_time_s,
                "generation_profile": (
                    f"fast-dllm-{FAST_DLLM_PARAMETERS}"
                    if MODELS[model_name].kind == "fast_dllm"
                    else "standard-greedy-json-only"
                ),
                "run_fingerprint": RUN_FINGERPRINT,
            }
            try:
                result = {
                    **base, **adapter.generate(row.prompt, row.task),
                    "status": "ok", "error": None,
                }
            except Exception as exc:
                result = {**base, "status": "error", "error": repr(exc)}
                new_rows.append(result)
                save_cache(pd.concat([cached, pd.DataFrame(new_rows)]), path)
                raise
            new_rows.append(result)
            if run_index % FLUSH_EVERY == 0 or run_index == len(remaining):
                save_cache(pd.concat([cached, pd.DataFrame(new_rows)]), path)
                print(f"  saved {run_index}/{len(remaining)}")
        return pd.read_parquet(path)
    finally:
        if adapter is not None:
            unload(adapter)
        else:
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

repair_prompts = (
    prompts.sort_values("prompt_id").groupby("task", group_keys=False)
    .head(FAST_REPAIR_PROMPTS_PER_TASK)
)
if RUN_FAST_DLLM_REPAIR_PILOT:
    benchmark_model("fast-dllm-v2-1.5b", repair_prompts)

fast_dllm_audit, fast_dllm_audit_passed = audit_fast_dllm_cache()
if RUN_EXPANDED_BENCHMARK and not fast_dllm_audit_passed:
    raise RuntimeError(
        "Fast-dLLM repair pilot did not pass. Inspect the audit before expansion."
    )
if not RUN_EXPANDED_BENCHMARK:
    raise RuntimeError(
        "Audit-only mode completed. Inspect responses, then enable expansion."
    )

for model_name in MODELS_TO_RUN:
    benchmark_model(model_name)

fast-dllm-v2-1.5b: 900 cached, 0 remaining


,prompts,parse_rate,strict_format_rate,exact_match,degenerate_rate,token_cap_rate,mean_latency_s
task,,,,,,,
arc_challenge,300,0.987,0.930,0.707,0.0,0.003,0.290
gsm8k,300,0.970,0.447,0.140,0.0,0.043,1.485
mmlu,300,0.993,0.903,0.490,0.0,0.003,0.341


,passed
audit_check,
all_tasks_present,True
parse_rate_at_least_90pct,True
at_least_one_exact_match,True
no_degenerate_responses,True
token_cap_rate_at_most_25pct,True


,task,prompt_id,reference,prediction,parsed,strict_format,quality,output_tokens,generation_s,response
752,arc_challenge,562c15ff4ac88208,A,B,True,True,False,7,0.208125,"{""answer"": ""B""}"
803,arc_challenge,87c325055558cef5,B,B,True,True,True,7,0.182414,"{""answer"": ""B""}"
866,arc_challenge,ad21ba5fc2b6356c,D,D,True,True,True,7,0.278664,"{""answer"": ""D""}"
833,arc_challenge,b84a70a7f29ba58f,D,B,True,True,False,12,0.314201,"{""answer"": ""B. long, strong limbs""}"
609,arc_challenge,f02c8930bc653faa,A,A,True,True,True,7,0.175680,"{""answer"": ""A""}"
258,gsm8k,3dfe08603d9cf952,4000,30000,True,True,False,11,0.354347,"{""answer"": ""30000""}"
142,gsm8k,4e81bd722890a68d,17,35,True,True,False,8,0.352144,"{""answer"": ""35""}"
174,gsm8k,55d22da9cd52c363,76,60,True,False,False,169,2.192740,"To determine the number of cakes Rose bought over the three days, we need to calculate the number of cakes she bought each day and then sum these amounts.\n\nFirst, let's calculate the number of cakes Rose bought on Monday:\n\[ \text{Cakes on Monday} = 4 \]\n\nNext, let's calculate the number of cakes Rose bought on Tuesday. She bought three times the number of cakes she bought on Monday:\n\[ \text{Cakes on Tuesday} = 3 \times 4 = 12 \]\n\nThen, let's calculate the number of cakes Rose bought on Wednesday. She bought five times the number of cakes she bought on Tuesday:\n\[ \text{Cakes on Wednesday} = 5 \times 12 = 60 \]\n\nNow, we sum the number of cakes bought on each"
208,gsm8k,76647bea6aa3985d,60,60,True,False,True,103,1.671458,"The bus traveled 60 miles per hour for 5 hours, so it traveled $60 \times 5 = 300$ miles.\n\nThe car traveled 30 miles per hour for 8 hours, so it traveled $30 \times 8 = 240$ miles.\n\nTherefore, the bus traveled $300 - 240 = 60$ miles farther than the car.\n\nSo, the answer is $\boxed{60}$ miles."
81,gsm8k,8b9aaeac92d6768d,39,6,True,False,False,130,2.594116,"Chris completes 12 problems on Monday night.\nOn Tuesday, he completes 3 times as many problems as he did on Monday, which is 3 * 12 = 36 problems.\nOn Wednesday, he completes one-quarter of the remaining math problems, which is 100 - 12 - 36 = 52 - 36 = 16 problems.\nTherefore, Chris has 100 - 12 - 36 - 16 = 22 - 16 = 6 math problems left to complete on Thursday.\n\n\boxed{answer"": ""6}"


Audit recommendation: PASS
qwen2.5-1.5b-ar: 900 cached, 0 remaining
fast-dllm-v2-1.5b: 900 cached, 0 remaining
qwen2.5-7b-4bit: 900 cached, 0 remaining


## 5. Parser-aware deterministic quality labels

The response evaluator records three different facts:

- **parsed:** an answer could be extracted;
- **strict format:** the entire response is exactly one JSON object containing
  only `answer`; and
- **quality:** the extracted answer matches the benchmark reference.

Keeping these separate prevents a formatting failure from being mistaken for a
reasoning failure during diagnosis. GSM8K numbers are canonicalized, so `36`,
`36.0`, and `036.000` compare equal. Multiple-choice tasks compare normalized
letters.

The cell also enforces a complete rectangular panel: every prompt must have one
successful measurement from every candidate before router training begins.

In [20]:
missing = [name for name in MODELS if not cache_path(name).exists()]
if missing:
    raise RuntimeError(f"Run the missing candidates before analysis: {missing}")

parts = [pd.read_parquet(cache_path(name)) for name in MODELS]
measurements = pd.concat(parts, ignore_index=True)
measurements = measurements.drop_duplicates(["prompt_id", "model"], keep="last")
measurements = measurements.loc[
    measurements.status.eq("ok")
    & measurements.run_fingerprint.eq(RUN_FINGERPRINT)
].merge(prompts, on=["prompt_id", "task"], validate="many_to_one")

# A missing candidate row would make safety labels and oracle choices invalid.
expected_panel = len(prompts) * len(MODELS)
if len(measurements) != expected_panel:
    completed = measurements.groupby("model").prompt_id.nunique().reindex(MODELS, fill_value=0)
    raise RuntimeError(
        f"Incomplete panel: {len(measurements)}/{expected_panel} rows.\n{completed}"
    )

measurements["prediction"] = [
    extract_answer(response, task)
    for response, task in zip(measurements.response, measurements.task)
]
measurements["parsed"] = measurements.prediction.notna()
measurements["strict_format"] = measurements.response.map(strict_json_answer).notna()
measurements["quality"] = measurements.prediction.eq(measurements.reference).astype(float)
measurements.to_parquet(ROOT / "data" / "measurements_v3.parquet", index=False)

summary = measurements.groupby(["model", "task"]).agg(
    prompts=("prompt_id", "size"), accuracy=("quality", "mean"),
    parse_rate=("parsed", "mean"), strict_format_rate=("strict_format", "mean"),
    generation_s=("generation_s", "mean"), output_tokens=("output_tokens", "mean"),
    tokens_per_second=("tokens_per_second", "mean"),
    peak_vram_gb=("peak_vram_gb", "max"), load_time_s=("load_time_s", "first"),
)
display(summary.round(3))

prompts  accuracy  parse_rate  \
model             task                                           
fast-dllm-v2-1.5b arc_challenge      300     0.707       0.987   
                  gsm8k              300     0.140       0.970   
                  mmlu               300     0.490       0.993   
qwen2.5-1.5b-ar   arc_challenge      300     0.683       1.000   
                  gsm8k              300     0.087       0.990   
                  mmlu               300     0.507       1.000   
qwen2.5-7b-4bit   arc_challenge      300     0.873       1.000   
                  gsm8k              300     0.203       1.000   
                  mmlu               300     0.680       1.000   

                                 strict_format_rate  generation_s  \
model             task                                              
fast-dllm-v2-1.5b arc_challenge               0.930         0.290   
                  gsm8k                       0.447         1.485   
                  mmlu                        0.903         0.341   
qwen2.5-1.5b-ar   arc_challenge               0.667         0.318   
                  gsm8k                       1.000         0.323   
                  mmlu                        0.590         0.327   
qwen2.5-7b-4bit   arc_challenge               1.000         0.577   
                  gsm8k                       1.000         0.629   
                  mmlu                        1.000         0.629   

                                 output_tokens  tokens_per_second  \
model             task                                              
fast-dllm-v2-1.5b arc_challenge         10.657             37.007   
                  gsm8k                 80.050             41.482   
                  mmlu                  11.217             32.972   
qwen2.5-1.5b-ar   arc_challenge          8.450             27.096   
                  gsm8k                  8.437             28.078   
                  mmlu                   8.850             27.695   
qwen2.5-7b-4bit   arc_challenge          7.000             12.262   
                  gsm8k                  8.257             13.217   
                  mmlu                   7.007             11.450   

                                 peak_vram_gb  load_time_s  
model             task                                      
fast-dllm-v2-1.5b arc_challenge         2.963       82.691  
                  gsm8k                 2.953       82.691  
                  mmlu                  3.084       82.691  
qwen2.5-1.5b-ar   arc_challenge         2.906       43.310  
                  gsm8k                 2.903       43.310  
                  mmlu                  2.938       43.310  
qwen2.5-7b-4bit   arc_challenge         5.356      309.953  
                  gsm8k                 5.347      309.953  
                  mmlu                  5.409      309.953

## 6. Prompt-level split, safety targets, and informative sampling

### Split discipline

Prompts—not prompt-model rows—are split 60%/20%/20%, stratified by task:

```text
train (540)       fit LoRA and the three heads
validation (180)  choose checkpoint, temperatures, and safety threshold
test (180)        final evaluation only
```

For each prompt, a model is observed as quality-safe when its score is within
`QUALITY_SAFETY_EPSILON` of the best candidate score. With binary exact match
and epsilon zero, every correct candidate is safe. If all candidates are wrong,
they tie and latency decides the offline oracle.

The strongest fallback is defined using training accuracy only. The sampler
upweights prompts where models disagree or where the fastest observed safe
candidate is not the fallback. These prompts contain more information about the
routing boundary than examples where all candidates behave identically.

In [22]:
MODEL_NAMES = list(MODELS)

def wide(column: str) -> pd.DataFrame:
    return measurements.pivot(
        index="prompt_id", columns="model", values=column
    ).reindex(columns=MODEL_NAMES)

table = prompts.set_index("prompt_id").join(wide("quality").add_prefix("q__"))
table = table.join(wide("generation_s").add_prefix("l__"))
table = table.join(wide("output_tokens").add_prefix("t__")).reset_index()

# Split by prompt index before constructing any learned router component.
train_val, test = train_test_split(
    table.index, test_size=.20, random_state=SEED, stratify=table.task
)
train, validation = train_test_split(
    train_val, test_size=.25, random_state=SEED + 1,
    stratify=table.loc[train_val, "task"],
)
masks = {
    "train": table.index.isin(train),
    "validation": table.index.isin(validation),
    "test": table.index.isin(test),
}
display(pd.DataFrame({
    name: table.loc[mask].groupby("task").size() for name, mask in masks.items()
}).fillna(0).astype(int))

Q = table[[f"q__{name}" for name in MODEL_NAMES]].to_numpy(float)
L = table[[f"l__{name}" for name in MODEL_NAMES]].to_numpy(float)
T = table[[f"t__{name}" for name in MODEL_NAMES]].to_numpy(float)
actual_best = Q.max(axis=1, keepdims=True)
safety_target = Q >= actual_best - QUALITY_SAFETY_EPSILON
oracle_idx = np.where(safety_target, L, np.inf).argmin(axis=1)
strongest_idx = int(Q[masks["train"]].mean(axis=0).argmax())
fastest_idx = int(L[masks["train"]].mean(axis=0).argmin())

disagreement = Q.max(axis=1) != Q.min(axis=1)
nonfallback_oracle = oracle_idx != strongest_idx
# Informative examples appear more often, but their targets are never changed.
sampling_weight = (
    1.0 + 3.0 * nonfallback_oracle.astype(float)
    + 1.5 * disagreement.astype(float)
)

text = (
    "[TASK=" + table.task.astype(str) + "] "
    + "[SUBJECT=" + table.subject.astype(str) + "] "
    + "[CHOICES=" + table.num_choices.astype(str) + "] "
    + "[LENGTH_BIN=" + table.length_bin.astype(str) + "] "
    + table.prompt.astype(str)
).to_numpy()

print({
    "strongest_fallback": MODEL_NAMES[strongest_idx],
    "fastest_training_baseline": MODEL_NAMES[fastest_idx],
    "train_prompts": int(masks["train"].sum()),
    "train_nonfallback_oracle_rate": float(nonfallback_oracle[masks["train"]].mean()),
    "train_disagreement_rate": float(disagreement[masks["train"]].mean()),
})

,train,validation,test
task,,,
arc_challenge,180,60,60
gsm8k,180,60,60
mmlu,180,60,60


{'strongest_fallback': 'qwen2.5-7b-4bit', 'fastest_training_baseline': 'qwen2.5-1.5b-ar', 'train_prompts': 540, 'train_nonfallback_oracle_rate': 0.8814814814814815, 'train_disagreement_rate': 0.37407407407407406}


## 7. Decision-aligned ModernBERT router with LoRA

### Why LoRA?

Fully fine-tuning a 149M-parameter encoder on 540 training prompts is likely to
overfit. Completely freezing it prevents the representation from adapting to
this routing task. LoRA provides a middle ground: the pretrained weights stay
frozen while small rank-8 matrices learn task-specific updates.

PEFT wraps ModernBERT for `FEATURE_EXTRACTION` and inserts LoRA into every linear
layer (`target_modules="all-linear"`). Only LoRA matrices and the router heads
receive gradients.

### Three prediction heads

| Head | Target | Loss | Used at routing time? |
|---|---|---|---|
| Safety | Is this model within the observed quality margin? | Balanced BCE | Yes |
| Latency | `log(1 + generation_s)` | Smooth-L1 | Yes |
| Tokens | `log(1 + output_tokens)` | Smooth-L1 | Diagnostic auxiliary task |

Attention-mask-aware mean pooling turns token representations into one prompt
vector. The token head regularizes the shared representation because output
length is strongly related to latency, while direct latency remains the actual
selection signal.

### Decision-aligned checkpoint selection

After every epoch, validation predictions are routed across the threshold grid.
A checkpoint that satisfies 98% quality retention and saves latency outranks a
checkpoint with a lower generic loss but worse routing decisions. Early stopping
therefore optimizes the downstream decision, not merely prediction error.

In [23]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

safety_targets = torch.tensor(safety_target.astype(float), dtype=torch.float32)
latency_targets = torch.tensor(np.log1p(L), dtype=torch.float32)
token_targets = torch.tensor(np.log1p(T), dtype=torch.float32)

# Positive weights counter the tendency to learn "always use the fallback".
train_safety_rate = safety_target[masks["train"]].mean(axis=0)
positive_weight = np.clip(
    (1.0 - train_safety_rate) / np.maximum(train_safety_rate, 1e-6),
    0.5, 5.0,
)
safety_positive_weight = torch.tensor(positive_weight, dtype=torch.float32).to("cuda")

router_tokenizer = AutoTokenizer.from_pretrained(
    ROUTER_ENCODER_REPO, revision=ROUTER_ENCODER_REVISION
)

class QualitySafeRouter(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Load the immutable base encoder, then inject trainable LoRA matrices.
        base_encoder = AutoModel.from_pretrained(
            ROUTER_ENCODER_REPO, revision=ROUTER_ENCODER_REVISION,
            attn_implementation="sdpa",
        )
        lora_config = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            inference_mode=False,
            r=ROUTER_LORA_R,
            lora_alpha=ROUTER_LORA_ALPHA,
            lora_dropout=ROUTER_LORA_DROPOUT,
            target_modules=ROUTER_LORA_TARGET_MODULES,
            bias="none",
        )
        self.encoder = get_peft_model(base_encoder, lora_config)
        hidden_size = int(base_encoder.config.hidden_size)
        self.dropout = torch.nn.Dropout(0.10)
        self.safety_head = torch.nn.Linear(hidden_size, len(MODEL_NAMES))
        self.latency_head = torch.nn.Linear(hidden_size, len(MODEL_NAMES))
        self.token_head = torch.nn.Linear(hidden_size, len(MODEL_NAMES))

    def forward(self, **inputs):
        hidden = self.encoder(**inputs).last_hidden_state
        mask = inputs["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)
        pooled = self.dropout(pooled)
        return {
            "safety_logits": self.safety_head(pooled),
            "latency_log": self.latency_head(pooled),
            "token_log": self.token_head(pooled),
        }

router_load_started = time.perf_counter()
router_model = QualitySafeRouter().to("cuda")
router_load_time_s = time.perf_counter() - router_load_started
router_model.encoder.print_trainable_parameters()

def collate_router_batch(indices):
    indices = [int(index) for index in indices]
    encoded = router_tokenizer(
        [f"classification: {text[index]}" for index in indices],
        padding=True, truncation=True, max_length=MAX_INPUT_TOKENS,
        return_tensors="pt",
    )
    return torch.tensor(indices, dtype=torch.long), encoded

train_indices = np.flatnonzero(masks["train"]).tolist()
train_weights = torch.tensor(sampling_weight[train_indices], dtype=torch.double)
# Weighted sampling exposes the router more often to useful model disagreements.
sampler = WeightedRandomSampler(
    train_weights, num_samples=len(train_indices), replacement=True,
    generator=torch.Generator().manual_seed(SEED),
)
train_loader = DataLoader(
    train_indices, batch_size=ROUTER_BATCH_SIZE, sampler=sampler,
    collate_fn=collate_router_batch,
)
validation_loader = DataLoader(
    np.flatnonzero(masks["validation"]).tolist(), batch_size=ROUTER_BATCH_SIZE * 2,
    shuffle=False, collate_fn=collate_router_batch,
)

def router_batch_loss(outputs, indices):
    s_target = safety_targets[indices].to("cuda")
    l_target = latency_targets[indices].to("cuda")
    t_target = token_targets[indices].to("cuda")
    safety_loss = F.binary_cross_entropy_with_logits(
        outputs["safety_logits"].float(), s_target,
        pos_weight=safety_positive_weight,
    )
    latency_loss = F.smooth_l1_loss(outputs["latency_log"].float(), l_target)
    token_loss = F.smooth_l1_loss(outputs["token_log"].float(), t_target)
    total = (
        ROUTER_SAFETY_LOSS_WEIGHT * safety_loss
        + ROUTER_LATENCY_LOSS_WEIGHT * latency_loss
        + ROUTER_TOKEN_LOSS_WEIGHT * token_loss
    )
    return total, safety_loss, latency_loss, token_loss

@torch.inference_mode()
def predict_loader(loader):
    router_model.eval()
    index_parts, safety_parts, latency_parts, token_parts = [], [], [], []
    totals = np.zeros(4, dtype=float)
    examples = 0
    for indices, encoded in loader:
        encoded = encoded.to("cuda")
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
            outputs = router_model(**encoded)
            losses = router_batch_loss(outputs, indices)
        batch_size = len(indices)
        examples += batch_size
        totals += batch_size * np.array([float(loss.detach()) for loss in losses])
        index_parts.append(indices.numpy())
        safety_parts.append(outputs["safety_logits"].float().cpu().numpy())
        latency_parts.append(outputs["latency_log"].float().cpu().numpy())
        token_parts.append(outputs["token_log"].float().cpu().numpy())
    return {
        "indices": np.concatenate(index_parts),
        "safety_logits": np.concatenate(safety_parts),
        "latency_log": np.concatenate(latency_parts),
        "token_log": np.concatenate(token_parts),
        "losses": totals / examples,
    }

def select_subset(safe_probability, latency_prediction, threshold):
    eligible = safe_probability >= threshold
    eligible[:, strongest_idx] = True
    chosen = np.where(eligible, latency_prediction, np.inf).argmin(axis=1)
    no_safe_alternative = ~np.delete(eligible, strongest_idx, axis=1).any(axis=1)
    return chosen, no_safe_alternative

def subset_metrics(indices, chosen, overhead_s=0.0):
    indices = np.asarray(indices)
    row = np.arange(len(indices))
    chosen_q = Q[indices][row, chosen]
    chosen_l = L[indices][row, chosen] + np.asarray(overhead_s)
    strong_q = Q[indices, strongest_idx]
    strong_l = L[indices, strongest_idx]
    regret = Q[indices].max(axis=1) - chosen_q
    return {
        "accuracy": float(chosen_q.mean()),
        "quality_retention": float(chosen_q.mean() / max(strong_q.mean(), 1e-9)),
        "latency_s": float(chosen_l.mean()),
        "latency_reduction": float(1 - chosen_l.mean() / strong_l.mean()),
        "strongest_usage": float(np.mean(chosen == strongest_idx)),
        "constraint_violation_rate": float(np.mean(regret > QUALITY_SAFETY_EPSILON)),
    }

def checkpoint_route_score(prediction):
    indices = prediction["indices"]
    safe_probability = 1 / (1 + np.exp(-prediction["safety_logits"]))
    latency_prediction = np.maximum(1e-6, np.expm1(prediction["latency_log"]))
    candidates = []
    for threshold in SAFETY_THRESHOLD_GRID:
        chosen, _ = select_subset(safe_probability, latency_prediction, threshold)
        metrics = subset_metrics(indices, chosen)
        candidates.append((threshold, metrics))
    feasible = [
        item for item in candidates
        if item[1]["quality_retention"] >= MIN_QUALITY_RETENTION
        and item[1]["latency_reduction"] > 0
    ]
    if feasible:
        best = max(
            feasible,
            key=lambda item: (
                item[1]["latency_reduction"],
                -item[1]["constraint_violation_rate"],
            ),
        )
        return True, best[1]["latency_reduction"], best[0]
    return False, -float(prediction["losses"][0]), None

encoder_parameters = [
    parameter for parameter in router_model.encoder.parameters()
    if parameter.requires_grad
]
head_parameters = [
    parameter for name, parameter in router_model.named_parameters()
    if not name.startswith("encoder.")
]
parameter_groups = [{"params": head_parameters, "lr": ROUTER_HEAD_LR}]
if encoder_parameters:
    parameter_groups.insert(0, {"params": encoder_parameters, "lr": ROUTER_LORA_LR})
optimizer = torch.optim.AdamW(parameter_groups, weight_decay=ROUTER_WEIGHT_DECAY)
updates_per_epoch = math.ceil(len(train_loader) / ROUTER_GRADIENT_ACCUMULATION)
total_training_steps = updates_per_epoch * ROUTER_MAX_EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, round(total_training_steps * ROUTER_WARMUP_RATIO)),
    num_training_steps=total_training_steps,
)
scaler = torch.amp.GradScaler("cuda", enabled=COMPUTE_DTYPE == torch.float16)

trainable_names = {
    name for name, parameter in router_model.named_parameters()
    if parameter.requires_grad
}
history = []
best_key = (-1, -np.inf)
best_state = None
best_epoch = 0
epochs_without_improvement = 0
training_started = time.perf_counter()

for epoch in range(1, ROUTER_MAX_EPOCHS + 1):
    # train() activates LoRA dropout and head dropout; frozen base weights still
    # receive no gradients because PEFT marks only adapter weights trainable.
    router_model.train()
    optimizer.zero_grad(set_to_none=True)
    train_totals = np.zeros(4, dtype=float)
    train_examples = 0
    accumulation_count = 0
    for step, (indices, encoded) in enumerate(train_loader, 1):
        encoded = encoded.to("cuda")
        accumulation_count += 1
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
            losses = router_batch_loss(router_model(**encoded), indices)
            scaled_loss = losses[0] / ROUTER_GRADIENT_ACCUMULATION
        scaler.scale(scaled_loss).backward()
        should_step = (
            accumulation_count == ROUTER_GRADIENT_ACCUMULATION
            or step == len(train_loader)
        )
        if should_step:
            if accumulation_count != ROUTER_GRADIENT_ACCUMULATION:
                correction = ROUTER_GRADIENT_ACCUMULATION / accumulation_count
                for parameter in router_model.parameters():
                    if parameter.grad is not None:
                        parameter.grad.mul_(correction)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in router_model.parameters() if p.requires_grad],
                ROUTER_MAX_GRAD_NORM,
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            accumulation_count = 0
        batch_size = len(indices)
        train_examples += batch_size
        train_totals += batch_size * np.array([float(loss.detach()) for loss in losses])

    validation_prediction = predict_loader(validation_loader)
    feasible_checkpoint, route_score, route_threshold = checkpoint_route_score(
        validation_prediction
    )
    validation_losses = validation_prediction["losses"]
    candidate_key = (int(feasible_checkpoint), float(route_score))
    record = {
        "epoch": epoch,
        "train_loss": train_totals[0] / train_examples,
        "validation_loss": validation_losses[0],
        "validation_safety_loss": validation_losses[1],
        "validation_latency_loss": validation_losses[2],
        "validation_token_loss": validation_losses[3],
        "validation_route_feasible": feasible_checkpoint,
        "validation_route_score": route_score,
        "validation_route_threshold": route_threshold,
    }
    history.append(record)
    print({
        key: round(value, 4) if isinstance(value, (float, np.floating)) else value
        for key, value in record.items()
    })

    if candidate_key > best_key:
        best_key = candidate_key
        best_epoch = epoch
        state = router_model.state_dict()
        best_state = {
            name: value.detach().cpu().clone()
            for name, value in state.items()
            if name in trainable_names
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if (
            epoch >= ROUTER_MIN_EPOCHS
            and epochs_without_improvement >= ROUTER_EARLY_STOPPING_PATIENCE
        ):
            print(f"Early stopping after epoch {epoch}; restoring epoch {best_epoch}.")
            break

router_training_time_s = time.perf_counter() - training_started
assert best_state is not None
router_model.load_state_dict(best_state, strict=False)
training_history = pd.DataFrame(history)
training_history.to_csv(ROOT / "reports" / "router_training_history_v3.csv", index=False)
display(training_history.round(4))

trainable params: 1,689,600 || all params: 150,703,872 || trainable%: 1.1211
{'epoch': 1, 'train_loss': np.float64(0.6033), 'validation_loss': np.float64(0.4153), 'validation_safety_loss': np.float64(0.357), 'validation_latency_loss': np.float64(0.047), 'validation_token_loss': np.float64(0.2304), 'validation_route_feasible': False, 'validation_route_score': -0.4153, 'validation_route_threshold': None}
{'epoch': 2, 'train_loss': np.float64(0.4389), 'validation_loss': np.float64(0.4078), 'validation_safety_loss': np.float64(0.3703), 'validation_latency_loss': np.float64(0.0287), 'validation_token_loss': np.float64(0.1598), 'validation_route_feasible': False, 'validation_route_score': -0.4078, 'validation_route_threshold': None}
{'epoch': 3, 'train_loss': np.float64(0.4107), 'validation_loss': np.float64(0.3962), 'validation_safety_loss': np.float64(0.3632), 'validation_latency_loss': np.float64(0.0246), 'validation_token_loss': np.float64(0.1457), 'validation_route_feasible': False, 'va

,epoch,train_loss,validation_loss,validation_safety_loss,validation_latency_loss,validation_token_loss,validation_route_feasible,validation_route_score,validation_route_threshold
0,1,0.6033,0.4153,0.3570,0.0470,0.2304,False,-0.4153,NaN
1,2,0.4389,0.4078,0.3703,0.0287,0.1598,False,-0.4078,NaN
2,3,0.4107,0.3962,0.3632,0.0246,0.1457,False,-0.3962,NaN
3,4,0.4061,0.3889,0.3545,0.0262,0.1472,False,-0.3889,NaN
4,5,0.3772,0.4003,0.3678,0.0242,0.1435,True,0.0038,0.8
5,6,0.3965,0.3802,0.3466,0.0262,0.1398,False,-0.3802,NaN
6,7,0.3977,0.3934,0.3628,0.0224,0.1374,True,0.0018,0.8
7,8,0.3858,0.3849,0.3542,0.0229,0.1357,True,0.0018,0.8
8,9,0.3878,0.3845,0.3536,0.0230,0.1363,False,-0.3845,NaN
9,10,0.3863,0.3894,0.3564,0.0252,0.1410,False,-0.3894,NaN


## 8. Safety calibration and sealed evaluation

Raw neural-network logits are scores, not trustworthy probabilities. Validation
therefore learns one temperature per candidate:

```text
raw safety logit / learned temperature -> calibrated P(model is safe)
```

The selector evaluates a grid of safety thresholds. The strongest training
model is always available; among eligible candidates it chooses the lowest
predicted direct latency. Router inference is measured at batch size one and
added to candidate generation time.

If no validation threshold both retains at least 98% quality and improves
latency, the safe outcome is to disable the router. In that case the strongest
model is used directly and router overhead is correctly set to zero.

Only after temperature and threshold selection are frozen does the notebook
evaluate the sealed test split.

In [24]:
@torch.inference_mode()
def predict_all_batch_one(values):
    router_model.eval()

    def predict_one(value):
        encoded = router_tokenizer(
            f"classification: {value}", return_tensors="pt", truncation=True,
            max_length=MAX_INPUT_TOKENS,
        ).to("cuda")
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
            return router_model(**encoded)

    for value in values[:ROUTER_WARMUP_PROMPTS]:
        predict_one(str(value))
    torch.cuda.synchronize()

    safety_logits, latency_logs, token_logs = [], [], []
    overhead_s = np.zeros(len(values), dtype=float)
    for index, value in enumerate(values):
        torch.cuda.synchronize()
        started = time.perf_counter()
        outputs = predict_one(str(value))
        torch.cuda.synchronize()
        overhead_s[index] = time.perf_counter() - started
        safety_logits.append(outputs["safety_logits"].float().cpu().numpy())
        latency_logs.append(outputs["latency_log"].float().cpu().numpy())
        token_logs.append(outputs["token_log"].float().cpu().numpy())
    return (
        np.concatenate(safety_logits), np.concatenate(latency_logs),
        np.concatenate(token_logs), overhead_s,
    )

safety_logits, latency_log_pred, token_log_pred, router_overhead_s = (
    predict_all_batch_one(text)
)

# Fit one scalar on validation only; test labels never enter calibration.
def fit_temperature(logits, targets):
    logits_tensor = torch.tensor(logits, dtype=torch.float32)
    targets_tensor = torch.tensor(targets, dtype=torch.float32)
    log_temperature = torch.nn.Parameter(torch.zeros(()))
    optimizer_temperature = torch.optim.LBFGS(
        [log_temperature], lr=0.1, max_iter=100, line_search_fn="strong_wolfe"
    )

    def closure():
        optimizer_temperature.zero_grad()
        temperature = log_temperature.exp().clamp(0.05, 20.0)
        loss = F.binary_cross_entropy_with_logits(
            logits_tensor / temperature, targets_tensor
        )
        loss.backward()
        return loss

    optimizer_temperature.step(closure)
    return float(log_temperature.exp().detach().clamp(0.05, 20.0))

validation_mask = masks["validation"]
safety_temperatures = np.array([
    fit_temperature(
        safety_logits[validation_mask, model_index],
        safety_target[validation_mask, model_index].astype(float),
    )
    for model_index in range(len(MODEL_NAMES))
])
safety_probability = 1 / (
    1 + np.exp(-safety_logits / safety_temperatures[None, :])
)
latency_pred = np.maximum(1e-6, np.expm1(latency_log_pred))
token_pred = np.maximum(1.0, np.expm1(token_log_pred))

# Evaluate the actual routing rule, including measured online router overhead.
calibration_rows = []
validation_indices = np.flatnonzero(validation_mask)
for threshold in SAFETY_THRESHOLD_GRID:
    chosen, used_fallback = select_subset(
        safety_probability, latency_pred, threshold
    )
    row = {
        "safety_threshold": threshold,
        **subset_metrics(
            validation_indices, chosen[validation_mask],
            router_overhead_s[validation_mask],
        ),
        "fallback_rate": float(used_fallback[validation_mask].mean()),
    }
    calibration_rows.append(row)
calibration = pd.DataFrame(calibration_rows)
display(calibration.round(4))

feasible = calibration.query(
    "quality_retention >= @MIN_QUALITY_RETENTION and latency_reduction > 0"
)
if feasible.empty:
    warnings.warn(
        "No validation setting improved latency while meeting retention; "
        "deploying the strongest model without router overhead."
    )
    selected_threshold = None
    router_idx = np.full(len(table), strongest_idx)
    router_confidence = np.ones(len(table))
    router_used_fallback = np.ones(len(table), dtype=bool)
    router_active = False
    selected_overhead = None
else:
    best = feasible.sort_values(
        ["latency_reduction", "constraint_violation_rate"],
        ascending=[False, True],
    ).iloc[0]
    selected_threshold = float(best.safety_threshold)
    router_idx, router_used_fallback = select_subset(
        safety_probability, latency_pred, selected_threshold
    )
    router_confidence = safety_probability[np.arange(len(table)), router_idx]
    router_active = True
    selected_overhead = router_overhead_s

def metrics(mask, chosen, overhead=None):
    indices = np.flatnonzero(mask)
    selected = chosen[mask]
    overhead_values = (
        np.zeros(len(indices)) if overhead is None else np.asarray(overhead)[mask]
    )
    result = subset_metrics(indices, selected, overhead_values)
    result["router_overhead_s"] = float(overhead_values.mean())
    regret = Q[mask].max(axis=1) - Q[mask][np.arange(len(indices)), selected]
    result["p95_quality_regret"] = float(np.quantile(regret, .95, method="higher"))
    return result

strategies = {
    "strongest": (np.full(len(table), strongest_idx), None),
    "fastest": (np.full(len(table), fastest_idx), None),
    "oracle": (oracle_idx, None),
    "router": (router_idx, selected_overhead),
}
evaluation = pd.DataFrame({
    name: metrics(masks["test"], chosen, overhead)
    for name, (chosen, overhead) in strategies.items()
}).T
display(evaluation.round(4))
print({
    "router_active": router_active,
    "selected_safety_threshold": selected_threshold,
    "safety_temperatures": dict(zip(MODEL_NAMES, safety_temperatures.round(4))),
    "best_epoch": best_epoch,
    "router_load_time_s": round(router_load_time_s, 3),
    "router_training_time_s": round(router_training_time_s, 3),
    "mean_router_overhead_ms": round(1000 * router_overhead_s.mean(), 3),
    "trainable_parameters": sum(
        parameter.numel() for parameter in router_model.parameters()
        if parameter.requires_grad
    ),
})

,safety_threshold,accuracy,quality_retention,latency_s,latency_reduction,strongest_usage,constraint_violation_rate,fallback_rate
0,0.500,0.3444,0.5741,0.3936,0.3682,0.0111,0.3222,0.0000
1,0.600,0.3611,0.6019,0.3854,0.3814,0.0222,0.3056,0.0111
2,0.700,0.4333,0.7222,0.4310,0.3082,0.2000,0.2333,0.1889
3,0.800,0.5611,0.9352,0.6161,0.0111,0.7722,0.1056,0.7000
4,0.900,0.6000,1.0000,0.6753,-0.0840,1.0000,0.0667,1.0000
5,0.950,0.6000,1.0000,0.6753,-0.0840,1.0000,0.0667,1.0000
6,0.975,0.6000,1.0000,0.6753,-0.0840,1.0000,0.0667,1.0000


/tmp/ipykernel_1443/2878591319.py:95: UserWarning: No validation setting improved latency while meeting retention; deploying the strongest model without router overhead.
  warnings.warn(


,accuracy,quality_retention,latency_s,latency_reduction,strongest_usage,constraint_violation_rate,router_overhead_s,p95_quality_regret
strongest,0.6389,1.0000,0.6041,0.0000,1.0000,0.1222,0.0,1.0
fastest,0.4111,0.6435,0.3295,0.4545,0.0000,0.3500,0.0,1.0
oracle,0.7611,1.1913,0.3883,0.3573,0.1722,0.0000,0.0,0.0
router,0.6389,1.0000,0.6041,0.0000,1.0000,0.1222,0.0,1.0


{'router_active': False, 'selected_safety_threshold': None, 'safety_temperatures': {'qwen2.5-1.5b-ar': np.float64(0.5242), 'fast-dllm-v2-1.5b': np.float64(0.9257), 'qwen2.5-7b-4bit': np.float64(0.3278)}, 'best_epoch': 5, 'router_load_time_s': 3.563, 'router_training_time_s': 246.063, 'mean_router_overhead_ms': np.float64(53.042), 'trainable_parameters': 1696521}


In [25]:
test_mask = masks["test"]
test_indices = np.flatnonzero(test_mask)
test_pick = router_idx[test_mask]
report = table.loc[test_mask, [
    "prompt_id", "task", "subject", "prompt", "reference"
]].copy()
report["selected_model"] = [MODEL_NAMES[index] for index in test_pick]
report["quality"] = Q[test_mask][np.arange(len(test_indices)), test_pick]
report["generation_s"] = L[test_mask][np.arange(len(test_indices)), test_pick]
report["router_overhead_s"] = (
    0.0 if selected_overhead is None else selected_overhead[test_mask]
)
report["latency_s"] = report.generation_s + report.router_overhead_s
report["quality_regret"] = Q[test_mask].max(axis=1) - report.quality.to_numpy()
report["safety_probability"] = router_confidence[test_mask]
report["used_fallback"] = router_used_fallback[test_mask]
report.to_parquet(ROOT / "reports" / "test_decisions_v3.parquet", index=False)

calibration.to_csv(ROOT / "reports" / "selector_calibration_v3.csv", index=False)
evaluation.to_csv(ROOT / "reports" / "evaluation_v3.csv")

calibration_diagnostics = []
latency_diagnostics = []
for model_index, model_name in enumerate(MODEL_NAMES):
    target = safety_target[test_mask, model_index].astype(float)
    probability = safety_probability[test_mask, model_index]
    calibration_diagnostics.append({
        "model": model_name,
        "temperature": safety_temperatures[model_index],
        "test_brier_score": brier_score_loss(target, probability),
        "mean_predicted_safety": probability.mean(),
        "observed_safety": target.mean(),
    })
    latency_diagnostics.append({
        "model": model_name,
        "test_mae_s": mean_absolute_error(
            L[test_mask, model_index], latency_pred[test_mask, model_index]
        ),
        "test_r2": r2_score(
            L[test_mask, model_index], latency_pred[test_mask, model_index]
        ),
        "token_mae": mean_absolute_error(
            T[test_mask, model_index], token_pred[test_mask, model_index]
        ),
    })
calibration_diagnostics = pd.DataFrame(calibration_diagnostics).set_index("model")
latency_diagnostics = pd.DataFrame(latency_diagnostics).set_index("model")
display(calibration_diagnostics.round(4))
display(latency_diagnostics.round(4))

rng = np.random.default_rng(SEED)
router_q = report.quality.to_numpy()
router_l = report.latency_s.to_numpy()
strong_q = Q[test_mask, strongest_idx]
strong_l = L[test_mask, strongest_idx]
bootstrap = []
for _ in range(2000):
    sample = rng.integers(0, len(report), len(report))
    bootstrap.append([
        router_q[sample].mean() - strong_q[sample].mean(),
        1 - router_l[sample].mean() / strong_l[sample].mean(),
    ])
confidence_intervals = pd.DataFrame(
    bootstrap, columns=["accuracy_delta", "latency_reduction"]
).quantile([.025, .5, .975])
display(confidence_intervals.round(4))
display(pd.crosstab(
    report.task, report.selected_model, normalize="index"
).round(3))

# Export enough information to reconstruct the prompt encoder, LoRA adapter,
# three heads, calibration, and final selector without rerunning training.
artifact_manifest = {
    **RUN_MANIFEST,
    "artifact_schema_version": 1,
    "model_names": MODEL_NAMES,
    "strongest_model": MODEL_NAMES[strongest_idx],
    "router_active": router_active,
    "safety_threshold": selected_threshold,
    "safety_temperatures": dict(zip(MODEL_NAMES, safety_temperatures.tolist())),
    "best_epoch": best_epoch,
    "router_heads": ["safety_logits", "latency_log", "token_log"],
    "adaptation": "lora",
    "lora_config": ROUTER_CONTRACT,
    "router_input_template": (
        "[TASK={task}] [SUBJECT={subject}] [CHOICES={num_choices}] "
        "[LENGTH_BIN={length_bin}] {prompt}"
    ),
}
artifact_dir = ROOT / "artifacts" / (
    f"{CACHE_TAG}__{ROUTER_FINGERPRINT[:12]}"
)
artifact_dir.mkdir(parents=True, exist_ok=True)
# PEFT writes only the compact adapter configuration and adapter weights.
router_model.encoder.save_pretrained(artifact_dir / "lora_adapter")
head_state = {
    name: value.detach().cpu() for name, value in router_model.state_dict().items()
    if not name.startswith("encoder.")
}
torch.save(head_state, artifact_dir / "router_heads.pt")
router_tokenizer.save_pretrained(artifact_dir / "tokenizer")
(artifact_dir / "router_manifest.json").write_text(
    json.dumps(artifact_manifest, indent=2), encoding="utf-8"
)
print("Saved artifacts to", artifact_dir)

,temperature,test_brier_score,mean_predicted_safety,observed_safety
model,,,,
qwen2.5-1.5b-ar,0.5242,0.2470,0.7073,0.6500
fast-dllm-v2-1.5b,0.9257,0.1954,0.7461,0.7389
qwen2.5-7b-4bit,0.3278,0.1137,0.9353,0.8778


,test_mae_s,test_r2,token_mae
model,,,
qwen2.5-1.5b-ar,0.1024,-0.5679,1.7064
fast-dllm-v2-1.5b,0.4584,0.1849,22.8987
qwen2.5-7b-4bit,0.1082,-1.4918,0.8778


,accuracy_delta,latency_reduction
0.025,0.0,0.0
0.500,0.0,0.0
0.975,0.0,0.0


selected_model,qwen2.5-7b-4bit
task,
arc_challenge,1.0
gsm8k,1.0
mmlu,1.0


Saved artifacts to /content/drive/MyDrive/llm_router_v3/artifacts/d2044a470f45f714__1e74d980bca9


## 9. Interactive error analysis

Aggregate metrics answer whether the router works; individual examples explain
why. The inspector lets you move through every prompt and compare:

- the chosen candidate and whether fallback was required;
- observed quality and observed safety;
- calibrated safety probability;
- measured versus predicted generation latency;
- strict JSON compliance; and
- every raw candidate response.

Train and validation examples are diagnostic. Only rows marked **TEST**
contribute to the sealed result.

In [26]:
import ipywidgets as widgets

split_name = np.full(len(table), "", dtype=object)
for name, mask in masks.items():
    split_name[mask] = name

all_rows = np.arange(len(table))
inspection = table[[
    "prompt_id", "task", "subject", "prompt", "reference"
]].copy()
inspection["split"] = split_name
inspection["selected_model"] = [MODEL_NAMES[index] for index in router_idx]
inspection["actual_quality"] = Q[all_rows, router_idx]
inspection["measured_generation_s"] = L[all_rows, router_idx]
inspection["predicted_generation_s"] = latency_pred[all_rows, router_idx]
inspection["safety_probability"] = router_confidence
inspection["used_fallback"] = router_used_fallback
inspection.to_parquet(
    ROOT / "reports" / "all_prompt_decisions_v3.parquet", index=False
)

responses_by_prompt = {
    prompt_id: frame.set_index("model")
    for prompt_id, frame in measurements.groupby("prompt_id", sort=False)
}
task_rows = {
    task: frame.sort_values("prompt_id").reset_index(drop=True)
    for task, frame in inspection.groupby("task", sort=True)
}

task_selector = widgets.Dropdown(
    options=list(task_rows), description="Task:",
    layout=widgets.Layout(width="330px"),
)
prompt_selector = widgets.IntSlider(
    value=1, min=1, max=N_PER_TASK, step=1, description="Prompt:",
    continuous_update=False, layout=widgets.Layout(width="520px"),
)
previous_button = widgets.Button(description="← Previous")
next_button = widgets.Button(description="Next →", button_style="primary")
inspector_output = widgets.Output()

def format_metric(value, digits=3):
    return "—" if pd.isna(value) else f"{float(value):.{digits}f}"

def render_prompt_card(*_):
    frame = task_rows[task_selector.value]
    position = min(prompt_selector.value - 1, len(frame) - 1)
    row = frame.iloc[position]
    candidates = responses_by_prompt[row.prompt_id]
    selected = row.selected_model
    table_index = int(table.index[table.prompt_id.eq(row.prompt_id)][0])

    comparison_rows, response_sections = [], []
    for model_index, model_name in enumerate(MODEL_NAMES):
        measured = candidates.loc[model_name]
        chosen_class = "chosen" if model_name == selected else ""
        marker = "✓ Selected" if model_name == selected else ""
        actual_safe = bool(safety_target[table_index, model_index])
        comparison_rows.append(
            f'<tr class="{chosen_class}"><td><strong>{html_lib.escape(model_name)}</strong><br>'
            f'<span class="selected-marker">{marker}</span></td>'
            f'<td>{int(measured.quality)}</td><td>{actual_safe}</td>'
            f'<td>{format_metric(safety_probability[table_index, model_index])}</td>'
            f'<td>{format_metric(measured.generation_s)} s</td>'
            f'<td>{format_metric(latency_pred[table_index, model_index])} s</td>'
            f'<td>{bool(measured.strict_format)}</td></tr>'
        )
        response_sections.append(
            f'<details {"open" if model_name == selected else ""}>'
            f'<summary>{"★ " if model_name == selected else ""}'
            f'{html_lib.escape(model_name)} response</summary>'
            f'<pre>{html_lib.escape(str(measured.response))}</pre></details>'
        )

    card = f"""
    <style>
      .router-card {{font-family:Inter,system-ui,sans-serif;border:1px solid #dbe3ef;
        border-radius:18px;padding:22px;background:#f8fbff;color:#172033}}
      .router-card .top {{display:flex;gap:10px;flex-wrap:wrap;margin-bottom:14px}}
      .router-card .pill {{padding:5px 10px;border-radius:999px;background:#e8efff;
        color:#294fb5;font-size:12px;font-weight:700}}
      .router-card .model {{background:#dff7ec;color:#116149}}
      .router-card .prompt {{white-space:pre-wrap;background:white;border-left:4px solid #668cff;
        padding:14px;border-radius:8px;margin:12px 0}}
      .router-card table {{border-collapse:collapse;width:100%;background:white;margin:14px 0}}
      .router-card th,.router-card td {{padding:9px;border-bottom:1px solid #e8edf5;text-align:left}}
      .router-card th {{font-size:11px;text-transform:uppercase;color:#59677c}}
      .router-card tr.chosen {{background:#edf9f4}}
      .router-card .selected-marker {{color:#14805e;font-size:11px;font-weight:700}}
      .router-card details {{background:white;border:1px solid #e5eaf2;border-radius:10px;
        padding:10px 12px;margin-top:8px}}
      .router-card pre {{white-space:pre-wrap;max-height:320px;overflow:auto}}
    </style>
    <div class="router-card">
      <div class="top"><span class="pill">{html_lib.escape(str(row.task))}</span>
        <span class="pill">{html_lib.escape(str(row.split)).upper()}</span>
        <span class="pill model">Selected: {html_lib.escape(selected)}</span></div>
      <h3>Prompt {position + 1} of {len(frame)}</h3>
      <div class="prompt">{html_lib.escape(str(row.prompt))}</div>
      <p><strong>Reference:</strong> {html_lib.escape(str(row.reference))} ·
        <strong>Selected safety probability:</strong> {format_metric(row.safety_probability)} ·
        <strong>Fallback:</strong> {"Yes" if bool(row.used_fallback) else "No"}</p>
      <table><thead><tr><th>Candidate</th><th>Quality</th><th>Actually safe</th>
        <th>P(safe)</th><th>Measured latency</th><th>Predicted latency</th>
        <th>Strict JSON</th></tr></thead><tbody>{''.join(comparison_rows)}</tbody></table>
      {''.join(response_sections)}
    </div>
    """
    with inspector_output:
        clear_output(wait=True)
        display(HTML(card))

def reset_task(change):
    prompt_selector.max = len(task_rows[change["new"]])
    prompt_selector.value = 1
    render_prompt_card()

task_selector.observe(reset_task, names="value")
prompt_selector.observe(render_prompt_card, names="value")
previous_button.on_click(
    lambda _: setattr(prompt_selector, "value", max(1, prompt_selector.value - 1))
)
next_button.on_click(
    lambda _: setattr(
        prompt_selector, "value", min(prompt_selector.max, prompt_selector.value + 1)
    )
)
display(widgets.VBox([
    widgets.HBox([task_selector, previous_button, next_button]),
    prompt_selector, inspector_output,
]))
render_prompt_card()

## 10. How to interpret the result

A positive v3 result requires all synchronized scope conditions:

1. test quality retention of at least 0.98;
2. positive latency reduction with a bootstrap interval that does not
   materially support a slowdown;
3. acceptable quality-constraint violations;
4. meaningful non-fallback routing in defensible prompt subgroups; and
5. enough use of at least one faster candidate to justify a separate worker.

### Important comparisons

- **Strongest:** always uses the best training-accuracy model.
- **Fastest:** always uses the lowest training-latency model.
- **Oracle:** uses observed test outcomes and is an unattainable upper bound.
- **Router:** uses only prompt text, metadata, learned predictions, and frozen
  validation calibration.

A large oracle gain with a small router gain means candidate complementarity
exists but remains difficult to predict. A narrow bootstrap interval around
zero means the experiment has not demonstrated a real latency benefit.

Compare v3 with v2 only after completing all 2,700 new fingerprinted
generations. JSON-only prompts intentionally invalidate v2 caches. Results are
controlled warm batch-size-one inference on one Colab GPU—not production
serving latency. Production savings require already-loaded model workers.